# Issue #15 Phase 2: Locked 2024 Preflight Reproduction Gate

This notebook contains only the safety checks and locked 2024 reproduction gate required before the 2025 out-of-time holdout may be accessed.

**Safety boundary:** the 2025 CSV is not opened, parsed, previewed, summarized, or evaluated here. This notebook does not contain a cell that loads it. No 2025 metrics are calculated. The saved pipeline is used only through its fitted inference paths; neither `fit` nor `fit_transform` is called.

Only aggregate counts and metrics are displayed. Complaint narratives, normalized-text hashes, row-level predictions, and decision scores remain local and are neither displayed nor saved.

In [1]:
from pathlib import Path
import ast
import hashlib
import inspect
import platform
import re
import subprocess
import sys
import warnings

import joblib
import nbformat
import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.utils.validation import check_is_fitted


def find_project_root(start_path: Path) -> Path:
    current = start_path.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / ".git").exists() and (candidate / "reports" / "2025_validation_protocol.md").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root and committed validation protocol.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.routing_rules import AUTO_ROUTE, HUMAN_REVIEW, route_from_scores


NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / "07_2025_out_of_time_validation.ipynb"
PROTOCOL_PATH = PROJECT_ROOT / "reports" / "2025_validation_protocol.md"
MODEL_PATH = PROJECT_ROOT / "models" / "best_tfidf_classifier.joblib"
REFERENCE_2024_PATH = PROJECT_ROOT / "data" / "processed" / "cfpb_complaints_2024_cleaned.csv"
HOLDOUT_2025_PATH = PROJECT_ROOT / "data" / "raw" / "cfpb_complaints_2025_raw.csv"
ROUTING_SOURCE_PATH = PROJECT_ROOT / "src" / "routing_rules.py"

ALLOWED_BRANCHES = ("issue-15-2025-holdout-evaluation", "main")
BASELINE_COMMIT = "fddca32f466737901bbf95534d08030b32e9599e"
EXPECTED_HOLDOUT_2025_SIZE = 73_806_040
EXPECTED_HOLDOUT_2025_SHA256 = "b59d7842e786f00d6be26b7980a42f67474acb9040db293ddd3641204d25eb3a"
EXPECTED_VERSIONS = {
    "Python": "3.11.15",
    "Scikit-learn": "1.9.0",
    "Pandas": "3.0.3",
    "NumPy": "2.4.6",
}
EXPECTED_FILES = {
    "Locked model": {
        "path": MODEL_PATH,
        "size": 3_392_109,
        "sha256": "4514e7e49e305e408e2eaaf296d8607b33e9320547685339eff263e4dda0c94a",
    },
    "Locked 2024 cleaned reference": {
        "path": REFERENCE_2024_PATH,
        "size": 54_908_639,
        "sha256": "b115eb0c4a20a881a6a45bfb74cb7d715a726537372baa7d68f09d657cdfd919",
    },
}
EXPECTED_CLASSES = (
    "Checking or savings account",
    "Credit card",
    "Credit reporting or other personal consumer reports",
    "Debt collection",
    "Money transfer, virtual currency, or money service",
    "Mortgage",
    "Student loan",
    "Vehicle loan or lease",
)
EXPECTED_MODELING_ROWS = 33_042
EXPECTED_DEVELOPMENT_ROWS = 26_433
EXPECTED_FINAL_TEST_ROWS = 6_609
OUTER_SPLITS = 5
FINAL_TEST_FOLD = 0
RANDOM_STATE = 42
MIN_TOP_SCORE = 0.08
MIN_SCORE_MARGIN = 0.73

EXPECTED_CLASSIFICATION = {
    "Accuracy": 0.8712,
    "Macro precision": 0.7734,
    "Macro recall": 0.7621,
    "Macro F1": 0.7671,
    "Weighted precision": 0.8721,
    "Weighted recall": 0.8712,
    "Weighted F1": 0.8715,
}
EXPECTED_ROUTING = {
    "Auto-routed rows": 5_092,
    "Human-review rows": 1_517,
    "Auto-routing coverage": 0.7705,
    "Human-review rate": 0.2295,
    "Auto-routed accuracy": 0.9503,
    "Auto-routed misroute rate": 0.0497,
}

check_records = []


def record_check(name: str, passed: bool, observed: object, expected: object) -> None:
    check_records.append(
        {
            "check": name,
            "status": "PASS" if bool(passed) else "FAIL",
            "observed": str(observed),
            "expected": str(expected),
        }
    )


def stop_if_failed(section: str) -> None:
    failures = [row for row in check_records if row["status"] == "FAIL"]
    if failures:
        display(pd.DataFrame(failures))
        raise RuntimeError(
            f"{section} failed. Stop before accessing the 2025 holdout; do not change the model or protocol."
        )


print(f"Project root located: {PROJECT_ROOT}")
print("2025 holdout access: DISABLED for this notebook")
print("Output policy: aggregate checks and metrics only")

Project root located: E:\MGA\ITEC6740\Final-Project\financial-complaint-auto-routing-nlp
2025 holdout access: DISABLED for this notebook
Output policy: aggregate checks and metrics only


## 1. Safety, branch, protocol, and environment checks

The notebook first verifies its execution boundary and exact artifact-compatible environment. It also confirms that the current protocol file is the same Git blob recorded on `origin/main`.

In [2]:
def run_git(*args: str, check: bool = True) -> subprocess.CompletedProcess[str]:
    return subprocess.run(
        ["git", *args],
        cwd=PROJECT_ROOT,
        check=check,
        capture_output=True,
        text=True,
    )


current_branch = run_git("branch", "--show-current").stdout.strip()
record_check("Current branch is allowed", current_branch in ALLOWED_BRANCHES, current_branch, ALLOWED_BRANCHES)

protocol_relative = PROTOCOL_PATH.relative_to(PROJECT_ROOT).as_posix()
remote_protocol_blob = run_git("rev-parse", f"origin/main:{protocol_relative}").stdout.strip()
local_protocol_blob = run_git("hash-object", protocol_relative).stdout.strip()
record_check(
    "Protocol committed and present on origin/main",
    local_protocol_blob == remote_protocol_blob,
    local_protocol_blob,
    remote_protocol_blob,
)

baseline_is_ancestor = run_git("merge-base", "--is-ancestor", BASELINE_COMMIT, "HEAD", check=False).returncode == 0
record_check(
    "Locked baseline commit is an ancestor of HEAD",
    baseline_is_ancestor,
    baseline_is_ancestor,
    True,
)

observed_versions = {
    "Python": platform.python_version(),
    "Scikit-learn": sklearn.__version__,
    "Pandas": pd.__version__,
    "NumPy": np.__version__,
}
for component, expected_version in EXPECTED_VERSIONS.items():
    observed_version = observed_versions[component]
    record_check(
        f"{component} version",
        observed_version == expected_version,
        observed_version,
        expected_version,
    )

record_check("2025 holdout access enabled", False is False, False, False)
stop_if_failed("Safety and environment checks")

display(
    pd.DataFrame(
        [{"component": key, "observed": value, "expected": EXPECTED_VERSIONS[key]} for key, value in observed_versions.items()]
    )
)
print("Safety, Git, and environment checks passed.")

,component,observed,expected
0,Python,3.11.15,3.11.15
1,Scikit-learn,1.9.0,1.9.0
2,Pandas,3.0.3,3.0.3
3,NumPy,2.4.6,2.4.6


Safety, Git, and environment checks passed.


In [3]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


integrity_rows = []
for label, specification in EXPECTED_FILES.items():
    path = specification["path"]
    exists = path.is_file()
    record_check(f"{label} path exists", exists, exists, True)
    if not exists:
        continue

    observed_size = path.stat().st_size
    observed_sha256 = sha256_file(path)
    size_matches = observed_size == specification["size"]
    hash_matches = observed_sha256 == specification["sha256"]
    record_check(f"{label} file size", size_matches, observed_size, specification["size"])
    record_check(f"{label} SHA-256", hash_matches, observed_sha256, specification["sha256"])
    integrity_rows.append(
        {
            "file": path.relative_to(PROJECT_ROOT).as_posix(),
            "size_bytes": observed_size,
            "size_status": "PASS" if size_matches else "FAIL",
            "sha256": observed_sha256,
            "sha256_status": "PASS" if hash_matches else "FAIL",
        }
    )

stop_if_failed("Locked file-integrity checks")
display(pd.DataFrame(integrity_rows))
print("Only the locked model and 2024 cleaned reference were fingerprinted in this notebook.")

,file,size_bytes,size_status,sha256,sha256_status
0,models/best_tfidf_classifier.joblib,3392109,PASS,4514e7e49e305e408e2eaaf296d8607b33e93205476853...,PASS
1,data/processed/cfpb_complaints_2024_cleaned.csv,54908639,PASS,b115eb0c4a20a881a6a45bfb74cb7d715a726537372baa...,PASS


Only the locked model and 2024 cleaned reference were fingerprinted in this notebook.


In [4]:
with warnings.catch_warnings(record=True) as load_warnings:
    warnings.simplefilter("always")
    pipeline = joblib.load(MODEL_PATH)

record_check("Model compatibility warnings", len(load_warnings) == 0, len(load_warnings), 0)
record_check("Loaded object is sklearn Pipeline", isinstance(pipeline, Pipeline), type(pipeline).__name__, "Pipeline")

try:
    check_is_fitted(pipeline)
    fitted_pipeline = True
except Exception:
    fitted_pipeline = False
record_check("Pipeline is fitted", fitted_pipeline, fitted_pipeline, True)

expected_step_names = ["tfidf", "classifier"]
observed_step_names = list(pipeline.named_steps) if isinstance(pipeline, Pipeline) else []
record_check("Pipeline step names", observed_step_names == expected_step_names, observed_step_names, expected_step_names)

tfidf = pipeline.named_steps["tfidf"]
classifier = pipeline.named_steps["classifier"]
record_check("TF-IDF step type", isinstance(tfidf, TfidfVectorizer), type(tfidf).__name__, "TfidfVectorizer")
record_check("Classifier step type", isinstance(classifier, LinearSVC), type(classifier).__name__, "LinearSVC")

expected_tfidf_params = {
    "input": "content",
    "encoding": "utf-8",
    "decode_error": "strict",
    "strip_accents": None,
    "lowercase": True,
    "preprocessor": None,
    "tokenizer": None,
    "analyzer": "word",
    "token_pattern": r"(?u)\b\w\w+\b",
    "ngram_range": (1, 2),
    "stop_words": None,
    "max_df": 1.0,
    "min_df": 2,
    "max_features": 50_000,
    "vocabulary": None,
    "binary": False,
    "dtype": np.float64,
    "norm": "l2",
    "use_idf": True,
    "smooth_idf": True,
    "sublinear_tf": False,
}
expected_svm_params = {
    "penalty": "l2",
    "loss": "squared_hinge",
    "dual": "auto",
    "tol": 0.0001,
    "C": 1.0,
    "multi_class": "ovr",
    "fit_intercept": True,
    "intercept_scaling": 1,
    "class_weight": "balanced",
    "verbose": 0,
    "random_state": 42,
    "max_iter": 10_000,
}


def parameter_values_match(observed: object, expected: object) -> bool:
    if expected is np.float64:
        return np.dtype(observed) == np.dtype(np.float64)
    return observed == expected


observed_tfidf_params = tfidf.get_params(deep=False)
tfidf_mismatches = {
    name: (observed_tfidf_params[name], expected)
    for name, expected in expected_tfidf_params.items()
    if not parameter_values_match(observed_tfidf_params[name], expected)
}
observed_svm_params = classifier.get_params(deep=False)
svm_mismatches = {
    name: (observed_svm_params[name], expected)
    for name, expected in expected_svm_params.items()
    if not parameter_values_match(observed_svm_params[name], expected)
}
record_check("Locked TF-IDF parameters", not tfidf_mismatches, tfidf_mismatches or "all matched", "all matched")
record_check("Locked Linear SVM parameters", not svm_mismatches, svm_mismatches or "all matched", "all matched")

fitted_vocabulary_length = len(tfidf.vocabulary_)
fitted_idf_length = len(tfidf.idf_)
record_check("Fitted TF-IDF vocabulary length", fitted_vocabulary_length == 50_000, fitted_vocabulary_length, 50_000)
record_check("Fitted TF-IDF IDF length", fitted_idf_length == 50_000, fitted_idf_length, 50_000)

pipeline_class_order = tuple(pipeline.classes_)
record_check("Fitted classes_ order", pipeline_class_order == EXPECTED_CLASSES, pipeline_class_order, EXPECTED_CLASSES)

routing_source = Path(inspect.getsourcefile(route_from_scores)).resolve()
record_check("Existing route_from_scores implementation", routing_source == ROUTING_SOURCE_PATH.resolve(), routing_source, ROUTING_SOURCE_PATH.resolve())
routing_signature = inspect.signature(route_from_scores)
required_routing_parameters = {"class_labels", "decision_scores", "min_top_score", "min_score_margin"}
record_check(
    "Routing function parameters",
    required_routing_parameters.issubset(routing_signature.parameters),
    tuple(routing_signature.parameters),
    tuple(sorted(required_routing_parameters)),
)
record_check("Locked top-score threshold", MIN_TOP_SCORE == 0.08, MIN_TOP_SCORE, 0.08)
record_check("Locked score-margin threshold", MIN_SCORE_MARGIN == 0.73, MIN_SCORE_MARGIN, 0.73)

stop_if_failed("Locked pipeline checks")
print("Locked fitted TF-IDF + Linear SVM pipeline checks passed.")
print(f"Pipeline steps: {observed_step_names}")
print(f"Fitted TF-IDF features: {fitted_vocabulary_length:,}")
print(f"Fitted classes: {len(pipeline_class_order)}")
print("Model loaded without compatibility warnings.")

Locked fitted TF-IDF + Linear SVM pipeline checks passed.
Pipeline steps: ['tfidf', 'classifier']
Fitted TF-IDF features: 50,000
Fitted classes: 8
Model loaded without compatibility warnings.


## 2. Reconstruct the locked 2024 reference partitions

The reconstruction uses only the fingerprinted 2024 cleaned reference. Hashes are temporary in-memory group identifiers and are never displayed or saved.

In [5]:
required_columns = ["clean_complaint_text", "product"]
source_df = pd.read_csv(REFERENCE_2024_PATH, usecols=required_columns)

missing_columns = sorted(set(required_columns).difference(source_df.columns))
missing_value_count = int(source_df[required_columns].isna().sum().sum()) if not missing_columns else -1
record_check("2024 required columns", not missing_columns, missing_columns or "present", "present")
record_check("2024 required-field missing values", missing_value_count == 0, missing_value_count, 0)
stop_if_failed("2024 cleaned-reference schema checks")
reference_2024_label_set = set(source_df["product"].dropna().astype("string").str.strip().unique())


def normalize_cleaned_text(value: object) -> str:
    return " ".join(str(value).strip().split())


def stable_text_hash(value: object) -> str:
    normalized = normalize_cleaned_text(value)
    if not normalized:
        raise ValueError("Normalized complaint text must not be empty.")
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


working_df = source_df[required_columns].copy()
working_df["normalized_text_hash"] = working_df["clean_complaint_text"].map(stable_text_hash)

label_counts_by_hash = working_df.groupby("normalized_text_hash", sort=False)["product"].nunique()
conflicting_hashes = set(label_counts_by_hash[label_counts_by_hash > 1].index)

locked_scope_df = working_df[working_df["product"].isin(EXPECTED_CLASSES)].copy()
locked_scope_df = locked_scope_df[
    ~locked_scope_df["normalized_text_hash"].isin(conflicting_hashes)
].copy()
remediated_df = locked_scope_df.drop_duplicates(
    subset=["normalized_text_hash", "product"],
    keep="first",
).reset_index(drop=True)

outer_splitter = StratifiedGroupKFold(
    n_splits=OUTER_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)
outer_folds = list(
    outer_splitter.split(
        remediated_df["clean_complaint_text"],
        remediated_df["product"],
        groups=remediated_df["normalized_text_hash"],
    )
)
development_indices, final_test_indices = outer_folds[FINAL_TEST_FOLD]
development_df = remediated_df.iloc[development_indices].reset_index(drop=True)
final_test_df = remediated_df.iloc[final_test_indices].reset_index(drop=True)

development_groups = set(development_df["normalized_text_hash"])
final_test_groups = set(final_test_df["normalized_text_hash"])
overlap_count = len(development_groups.intersection(final_test_groups))
development_classes = set(development_df["product"].unique())
final_test_classes = set(final_test_df["product"].unique())
expected_class_set = set(EXPECTED_CLASSES)

record_check("Corrected 2024 modeling rows", len(remediated_df) == EXPECTED_MODELING_ROWS, len(remediated_df), EXPECTED_MODELING_ROWS)
record_check("2024 modeling hashes are unique", remediated_df["normalized_text_hash"].is_unique, remediated_df["normalized_text_hash"].is_unique, True)
record_check("2024 development rows", len(development_df) == EXPECTED_DEVELOPMENT_ROWS, len(development_df), EXPECTED_DEVELOPMENT_ROWS)
record_check("2024 final internal-test rows", len(final_test_df) == EXPECTED_FINAL_TEST_ROWS, len(final_test_df), EXPECTED_FINAL_TEST_ROWS)
record_check("All eight classes in development", development_classes == expected_class_set, len(development_classes), len(expected_class_set))
record_check("All eight classes in final internal test", final_test_classes == expected_class_set, len(final_test_classes), len(expected_class_set))
record_check("Development/final-test normalized-text overlap", overlap_count == 0, overlap_count, 0)
stop_if_failed("Locked 2024 reconstruction")

display(
    pd.DataFrame(
        [
            {"aggregate": "Corrected modeling rows", "observed": len(remediated_df), "expected": EXPECTED_MODELING_ROWS},
            {"aggregate": "Development rows", "observed": len(development_df), "expected": EXPECTED_DEVELOPMENT_ROWS},
            {"aggregate": "Final internal-test rows", "observed": len(final_test_df), "expected": EXPECTED_FINAL_TEST_ROWS},
            {"aggregate": "Classes in each partition", "observed": len(development_classes), "expected": len(expected_class_set)},
            {"aggregate": "Normalized-text overlap", "observed": overlap_count, "expected": 0},
        ]
    )
)
print("Locked 2024 reference reconstruction passed.")

,aggregate,observed,expected
0,Corrected modeling rows,33042,33042
1,Development rows,26433,26433
2,Final internal-test rows,6609,6609
3,Classes in each partition,8,8
4,Normalized-text overlap,0,0


Locked 2024 reference reconstruction passed.


## 3. Reproduce locked 2024 classification results

The already fitted pipeline evaluates only the reconstructed final internal-test rows. No model training or parameter selection occurs.

In [6]:
y_final = final_test_df["product"].to_numpy()
final_predictions = pipeline.predict(final_test_df["clean_complaint_text"])

macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    y_final,
    final_predictions,
    labels=list(pipeline_class_order),
    average="macro",
    zero_division=0,
)
weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
    y_final,
    final_predictions,
    labels=list(pipeline_class_order),
    average="weighted",
    zero_division=0,
)
observed_classification = {
    "Accuracy": float(accuracy_score(y_final, final_predictions)),
    "Macro precision": float(macro_precision),
    "Macro recall": float(macro_recall),
    "Macro F1": float(macro_f1),
    "Weighted precision": float(weighted_precision),
    "Weighted recall": float(weighted_recall),
    "Weighted F1": float(weighted_f1),
}

classification_rows = []
for metric, expected_value in EXPECTED_CLASSIFICATION.items():
    observed_value = observed_classification[metric]
    passed = round(observed_value, 4) == expected_value
    record_check(f"2024 classification: {metric}", passed, f"{observed_value:.4f}", f"{expected_value:.4f}")
    classification_rows.append(
        {
            "metric": metric,
            "observed": f"{observed_value:.4f}",
            "expected": f"{expected_value:.4f}",
            "status": "PASS" if passed else "FAIL",
        }
    )

stop_if_failed("Locked 2024 classification reproduction")
display(pd.DataFrame(classification_rows))
print("All locked 2024 classification metrics reproduced to four decimal places.")

,metric,observed,expected,status
0,Accuracy,0.8712,0.8712,PASS
1,Macro precision,0.7734,0.7734,PASS
2,Macro recall,0.7621,0.7621,PASS
3,Macro F1,0.7671,0.7671,PASS
4,Weighted precision,0.8721,0.8721,PASS
5,Weighted recall,0.8712,0.8712,PASS
6,Weighted F1,0.8715,0.8715,PASS


All locked 2024 classification metrics reproduced to four decimal places.


## 4. Reproduce locked 2024 routing results

Routing uses the fitted pipeline's decision function and class order, the existing `route_from_scores` implementation, and the locked inclusive thresholds. Decision scores are model signals, not probabilities, and are never displayed or saved.

In [7]:
final_score_matrix = pipeline.decision_function(final_test_df["clean_complaint_text"])
score_shape_ok = final_score_matrix.shape == (EXPECTED_FINAL_TEST_ROWS, len(pipeline_class_order))
finite_scores = bool(np.isfinite(final_score_matrix).all())
record_check("Final-test decision-score matrix shape", score_shape_ok, final_score_matrix.shape, (EXPECTED_FINAL_TEST_ROWS, len(pipeline_class_order)))
record_check("Final-test decision scores are finite", finite_scores, finite_scores, True)
stop_if_failed("Locked 2024 decision-score checks")

final_decisions = [
    route_from_scores(
        pipeline_class_order,
        score_row,
        min_top_score=MIN_TOP_SCORE,
        min_score_margin=MIN_SCORE_MARGIN,
    )
    for score_row in final_score_matrix
]
final_auto_mask = np.asarray(
    [decision["routing_decision"] == AUTO_ROUTE for decision in final_decisions],
    dtype=bool,
)
final_review_mask = np.asarray(
    [decision["routing_decision"] == HUMAN_REVIEW for decision in final_decisions],
    dtype=bool,
)
routed_predictions = np.asarray([decision["predicted_label"] for decision in final_decisions], dtype=object)
prediction_alignment = bool(np.array_equal(routed_predictions, final_predictions))
routing_partition_complete = bool(np.all(final_auto_mask ^ final_review_mask))
record_check("Routing predictions align with pipeline predictions", prediction_alignment, prediction_alignment, True)
record_check("Every row has exactly one routing outcome", routing_partition_complete, routing_partition_complete, True)

final_is_correct = final_predictions == y_final
auto_routed_rows = int(final_auto_mask.sum())
human_review_rows = int(final_review_mask.sum())
auto_routing_coverage = auto_routed_rows / EXPECTED_FINAL_TEST_ROWS
human_review_rate = human_review_rows / EXPECTED_FINAL_TEST_ROWS
auto_routed_accuracy = float(final_is_correct[final_auto_mask].mean())
auto_routed_misroute_rate = 1.0 - auto_routed_accuracy

observed_routing = {
    "Auto-routed rows": auto_routed_rows,
    "Human-review rows": human_review_rows,
    "Auto-routing coverage": auto_routing_coverage,
    "Human-review rate": human_review_rate,
    "Auto-routed accuracy": auto_routed_accuracy,
    "Auto-routed misroute rate": auto_routed_misroute_rate,
}

routing_rows = []
for metric, expected_value in EXPECTED_ROUTING.items():
    observed_value = observed_routing[metric]
    if isinstance(expected_value, int):
        passed = observed_value == expected_value
        observed_display = f"{observed_value:,}"
        expected_display = f"{expected_value:,}"
    else:
        passed = round(observed_value, 4) == expected_value
        observed_display = f"{observed_value:.4f}"
        expected_display = f"{expected_value:.4f}"
    record_check(f"2024 routing: {metric}", passed, observed_display, expected_display)
    routing_rows.append(
        {
            "metric": metric,
            "observed": observed_display,
            "expected": expected_display,
            "status": "PASS" if passed else "FAIL",
        }
    )

stop_if_failed("Locked 2024 routing reproduction")
display(pd.DataFrame(routing_rows))
print("All locked 2024 routing counts and metrics reproduced.")

# Remove row-level labels, predictions, temporary hashes, routing decisions, and scores from memory.
del final_score_matrix, final_decisions, routed_predictions, final_auto_mask, final_review_mask, final_is_correct
del final_predictions, y_final, source_df, working_df, locked_scope_df, remediated_df
del development_df, final_test_df, conflicting_hashes
print("Temporary row-level and hash-bearing objects cleared from memory.")

,metric,observed,expected,status
0,Auto-routed rows,"5,092","5,092",PASS
1,Human-review rows,"1,517","1,517",PASS
2,Auto-routing coverage,0.7705,0.7705,PASS
3,Human-review rate,0.2295,0.2295,PASS
4,Auto-routed accuracy,0.9503,0.9503,PASS
5,Auto-routed misroute rate,0.0497,0.0497,PASS


All locked 2024 routing counts and metrics reproduced.
Temporary row-level and hash-bearing objects cleared from memory.


## 5. Final 2024 preflight summary

The final static safety audit verifies that the notebook contains no training call, contains exactly one CSV read targeting the locked 2024 reference variable, and contains no code reference to the locked 2025 filename. The complete PASS/FAIL table below contains only aggregate and configuration evidence.

In [8]:
with NOTEBOOK_PATH.open("r", encoding="utf-8") as notebook_handle:
    notebook_document = nbformat.read(notebook_handle, as_version=4)

code_cells = [cell.source for cell in notebook_document.cells if cell.cell_type == "code"]
forbidden_training_calls = []
csv_read_arguments = []
approved_holdout_hash_calls = []
holdout_parser_calls = []
parser_call_names = {"read_csv", "read_table", "read_fwf", "read_json", "read_parquet"}
for cell_index, source in enumerate(code_cells):
    syntax_tree = ast.parse(source)
    for node in ast.walk(syntax_tree):
        if not isinstance(node, ast.Call):
            continue
        if isinstance(node.func, ast.Attribute):
            call_name = node.func.attr
        elif isinstance(node.func, ast.Name):
            call_name = node.func.id
        else:
            call_name = ""
        if call_name in {"fit", "fit_transform"}:
            forbidden_training_calls.append((cell_index, call_name))
        if call_name == "read_csv" and node.args:
            csv_read_arguments.append(ast.unparse(node.args[0]))
        if call_name == "sha256_file" and node.args and ast.unparse(node.args[0]) == "HOLDOUT_2025_PATH":
            approved_holdout_hash_calls.append(cell_index)
        if call_name in parser_call_names and node.args and "HOLDOUT_2025_PATH" in ast.unparse(node.args[0]):
            holdout_parser_calls.append((cell_index, call_name))

record_check(
    "No fit or fit_transform call in notebook",
    not forbidden_training_calls,
    forbidden_training_calls or "none",
    "none",
)
record_check(
    "CSV reads use only the locked 2024 and 2025 sources",
    csv_read_arguments == ["REFERENCE_2024_PATH", "HOLDOUT_2025_PATH"],
    csv_read_arguments,
    ["REFERENCE_2024_PATH", "HOLDOUT_2025_PATH"],
)

record_check(
    "2025 access is limited to the approved hash and one locked CSV load",
    len(approved_holdout_hash_calls) == 1 and len(holdout_parser_calls) == 1 and holdout_parser_calls[0][1] == "read_csv",
    f"binary_hash_calls={len(approved_holdout_hash_calls)}, parser_calls={len(holdout_parser_calls)}",
    "binary_hash_calls=1, parser_calls=1",
)

save_call_names = {"to_csv", "to_parquet", "to_pickle", "to_json", "save", "savez", "dump"}
row_level_save_calls = []
for cell_index, source in enumerate(code_cells):
    syntax_tree = ast.parse(source)
    for node in ast.walk(syntax_tree):
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Attribute) and node.func.attr in save_call_names:
            row_level_save_calls.append((cell_index, node.func.attr))
record_check(
    "No row-level data save call in notebook",
    not row_level_save_calls,
    row_level_save_calls or "none",
    "none",
)

stop_if_failed("Final 2024 preflight safety audit")
preflight_table = pd.DataFrame(check_records)
display(preflight_table)

all_checks_passed = bool(preflight_table["status"].eq("PASS").all())
if not all_checks_passed:
    raise RuntimeError("2024 preflight gate FAILED. The 2025 holdout must remain unopened.")

print(f"Required checks passed: {len(preflight_table)}/{len(preflight_table)}")
print("2024 PREFLIGHT GATE: PASS")
print("The 2025 holdout remains unparsed and unevaluated; only the approved binary integrity check is authorized next.")

,check,status,observed,expected
0,Current branch is allowed,PASS,issue-15-2025-holdout-evaluation,"('issue-15-2025-holdout-evaluation', 'main')"
1,Protocol committed and present on origin/main,PASS,3f7e9b293227e68884ecc2e6e240c01cdaec8d00,3f7e9b293227e68884ecc2e6e240c01cdaec8d00
2,Locked baseline commit is an ancestor of HEAD,PASS,True,True
3,Python version,PASS,3.11.15,3.11.15
4,Scikit-learn version,PASS,1.9.0,1.9.0
5,Pandas version,PASS,3.0.3,3.0.3
6,NumPy version,PASS,2.4.6,2.4.6
7,2025 holdout access enabled,PASS,False,False
8,Locked model path exists,PASS,True,True
9,Locked model file size,PASS,3392109,3392109


Required checks passed: 59/59
2024 PREFLIGHT GATE: PASS
The 2025 holdout remains unparsed and unevaluated; only the approved binary integrity check is authorized next.


## 2025 Holdout File-Integrity Gate

This gate performs only the pre-open integrity verification committed in the validation protocol. The file is read solely as uninterpreted binary bytes to calculate SHA-256. No CSV parsing, column or row inspection, narrative or label inspection, preview, summary, prediction, or 2025 metric calculation occurs.

In [9]:
integrity_check_start = len(check_records)
expected_holdout_relative = Path("data") / "raw" / "cfpb_complaints_2025_raw.csv"
observed_holdout_relative = HOLDOUT_2025_PATH.relative_to(PROJECT_ROOT)
holdout_exists = HOLDOUT_2025_PATH.is_file()
holdout_path_matches = observed_holdout_relative == expected_holdout_relative

record_check(
    "2025 holdout path exists and matches protocol",
    holdout_exists and holdout_path_matches,
    observed_holdout_relative.as_posix(),
    expected_holdout_relative.as_posix(),
)

if holdout_exists:
    observed_holdout_size = HOLDOUT_2025_PATH.stat().st_size
    observed_holdout_sha256 = sha256_file(HOLDOUT_2025_PATH)
else:
    observed_holdout_size = None
    observed_holdout_sha256 = None

holdout_size_matches = observed_holdout_size == EXPECTED_HOLDOUT_2025_SIZE
holdout_sha256_matches = observed_holdout_sha256 == EXPECTED_HOLDOUT_2025_SHA256
record_check(
    "2025 holdout file size",
    holdout_size_matches,
    observed_holdout_size if observed_holdout_size is not None else "not computed",
    EXPECTED_HOLDOUT_2025_SIZE,
)
record_check(
    "2025 holdout SHA-256",
    holdout_sha256_matches,
    "matches locked protocol" if holdout_sha256_matches else "mismatch or not computed",
    "matches locked protocol",
)

stop_if_failed("2025 holdout file-integrity gate")
holdout_integrity_table = pd.DataFrame(check_records[integrity_check_start:])
display(holdout_integrity_table)

entry_check_summary = pd.DataFrame(check_records)
display(entry_check_summary)
all_entry_checks_passed = bool(entry_check_summary["status"].eq("PASS").all())
if not all_entry_checks_passed:
    raise RuntimeError("Phase 2 entry checks FAILED. Do not parse or evaluate the 2025 holdout.")

print("2025 holdout path check: PASS")
print("2025 holdout file-size check: PASS")
print("2025 holdout SHA-256 check: PASS")
print("Existing 2024 preflight checks passed: 59/59")
print(f"Final entry checks passed: {len(entry_check_summary)}/{len(entry_check_summary)}")
print("2025 HOLDOUT FILE-INTEGRITY GATE: PASS")
print("The 2025 file was read only as raw binary bytes; its CSV contents remain unparsed and unevaluated.")

,check,status,observed,expected
0,2025 holdout path exists and matches protocol,PASS,data/raw/cfpb_complaints_2025_raw.csv,data/raw/cfpb_complaints_2025_raw.csv
1,2025 holdout file size,PASS,73806040,73806040
2,2025 holdout SHA-256,PASS,matches locked protocol,matches locked protocol


,check,status,observed,expected
0,Current branch is allowed,PASS,issue-15-2025-holdout-evaluation,"('issue-15-2025-holdout-evaluation', 'main')"
1,Protocol committed and present on origin/main,PASS,3f7e9b293227e68884ecc2e6e240c01cdaec8d00,3f7e9b293227e68884ecc2e6e240c01cdaec8d00
2,Locked baseline commit is an ancestor of HEAD,PASS,True,True
3,Python version,PASS,3.11.15,3.11.15
4,Scikit-learn version,PASS,1.9.0,1.9.0
...,...,...,...,...
57,2025 access is limited to the approved hash an...,PASS,"binary_hash_calls=1, parser_calls=1","binary_hash_calls=1, parser_calls=1"
58,No row-level data save call in notebook,PASS,none,none
59,2025 holdout path exists and matches protocol,PASS,data/raw/cfpb_complaints_2025_raw.csv,data/raw/cfpb_complaints_2025_raw.csv
60,2025 holdout file size,PASS,73806040,73806040


2025 holdout path check: PASS
2025 holdout file-size check: PASS
2025 holdout SHA-256 check: PASS
Existing 2024 preflight checks passed: 59/59
Final entry checks passed: 62/62
2025 HOLDOUT FILE-INTEGRITY GATE: PASS
The 2025 file was read only as raw binary bytes; its CSV contents remain unparsed and unevaluated.


## 2025 Dataset Loading and Structural Validation

The byte-level integrity gate above must pass before this section runs. This section loads only the fingerprinted 2025 source and reports aggregate structural and label-scope information. It does not display complaint narratives, complaint identifiers, or row-level data.

In [10]:
if not all_entry_checks_passed or len(entry_check_summary) != 62:
    raise RuntimeError("The 62 Phase 2 entry checks must pass before the 2025 CSV is parsed.")

phase2_data_check_start = len(check_records)
raw_2025_df = pd.read_csv(HOLDOUT_2025_PATH, low_memory=False)

required_2025_columns = [
    "complaint_what_happened",
    "product",
    "date_received",
    "complaint_id",
]
missing_2025_columns = sorted(set(required_2025_columns).difference(raw_2025_df.columns))
record_check(
    "2025 required raw columns are present",
    not missing_2025_columns,
    missing_2025_columns or "all present",
    "all present",
)
stop_if_failed("2025 required-column validation")


def missing_or_blank(series: pd.Series) -> pd.Series:
    return series.astype("string").str.strip().fillna("").eq("")


narrative_blank_mask = missing_or_blank(raw_2025_df["complaint_what_happened"])
product_blank_mask = missing_or_blank(raw_2025_df["product"])
date_blank_mask = missing_or_blank(raw_2025_df["date_received"])
complaint_id_blank_mask = missing_or_blank(raw_2025_df["complaint_id"])

date_text = raw_2025_df["date_received"].astype("string").str.strip()
parsed_date_received = pd.to_datetime(date_text.mask(date_blank_mask, pd.NA), errors="coerce")
unparseable_date_mask = ~date_blank_mask & parsed_date_received.isna()
outside_2025_mask = parsed_date_received.notna() & parsed_date_received.dt.year.ne(2025)
minimum_date_received = parsed_date_received.min().date().isoformat()
maximum_date_received = parsed_date_received.max().date().isoformat()

complaint_id_text = raw_2025_df["complaint_id"].astype("string").str.strip()
duplicate_complaint_id_values = int(complaint_id_text.loc[~complaint_id_blank_mask].duplicated().sum())

observed_rows = len(raw_2025_df)
observed_columns = len(raw_2025_df.columns)
record_check("2025 raw row count", observed_rows == 50_000, observed_rows, 50_000)
record_check("2025 raw column count", observed_columns == 17, observed_columns, 17)
record_check("2025 minimum date_received", minimum_date_received == "2025-01-01", minimum_date_received, "2025-01-01")
record_check("2025 maximum date_received", maximum_date_received == "2025-12-31", maximum_date_received, "2025-12-31")
record_check("2025 rows outside calendar year", int(outside_2025_mask.sum()) == 0, int(outside_2025_mask.sum()), 0)
record_check("2025 unparseable nonblank dates", int(unparseable_date_mask.sum()) == 0, int(unparseable_date_mask.sum()), 0)
record_check("2025 duplicate complaint_id values", duplicate_complaint_id_values == 0, duplicate_complaint_id_values, 0)
record_check("2025 missing or blank narratives", int(narrative_blank_mask.sum()) == 0, int(narrative_blank_mask.sum()), 0)
record_check("2025 missing or blank product labels", int(product_blank_mask.sum()) == 0, int(product_blank_mask.sum()), 0)
record_check("2025 missing or blank date_received", int(date_blank_mask.sum()) == 0, int(date_blank_mask.sum()), 0)
record_check("2025 missing or blank complaint_id", int(complaint_id_blank_mask.sum()) == 0, int(complaint_id_blank_mask.sum()), 0)
stop_if_failed("2025 structural validation")

required_column_table = pd.DataFrame(
    {"required_column": required_2025_columns, "present": [column in raw_2025_df.columns for column in required_2025_columns]}
)
structural_validation_table = pd.DataFrame(
    [
        {"check": "Total rows", "observed": observed_rows, "expected": 50_000},
        {"check": "Total columns", "observed": observed_columns, "expected": 17},
        {"check": "Minimum date_received", "observed": minimum_date_received, "expected": "2025-01-01"},
        {"check": "Maximum date_received", "observed": maximum_date_received, "expected": "2025-12-31"},
        {"check": "Rows outside 2025", "observed": int(outside_2025_mask.sum()), "expected": 0},
        {"check": "Unparseable nonblank dates", "observed": int(unparseable_date_mask.sum()), "expected": 0},
        {"check": "Duplicate complaint_id values", "observed": duplicate_complaint_id_values, "expected": 0},
        {"check": "Missing/blank narratives", "observed": int(narrative_blank_mask.sum()), "expected": 0},
        {"check": "Missing/blank product labels", "observed": int(product_blank_mask.sum()), "expected": 0},
        {"check": "Missing/blank date_received", "observed": int(date_blank_mask.sum()), "expected": 0},
        {"check": "Missing/blank complaint_id", "observed": int(complaint_id_blank_mask.sum()), "expected": 0},
    ]
)
display(required_column_table)
display(structural_validation_table)

raw_product_text = raw_2025_df["product"].astype("string").str.strip().fillna("")
raw_product_label_counts = raw_product_text.replace("", "<missing or blank>").value_counts(dropna=False).rename_axis("product_label").reset_index(name="rows")
raw_product_labels = set(raw_product_text.loc[raw_product_text.ne("")].unique())
locked_class_set = set(EXPECTED_CLASSES)


def product_scope_status(label: str) -> str:
    if label in locked_class_set:
        return "locked eight-category scope"
    if label in reference_2024_label_set:
        return "familiar out-of-scope label"
    return "unfamiliar or changed label"


raw_product_label_counts["scope_status"] = raw_product_label_counts["product_label"].map(product_scope_status)
label_scope_summary = raw_product_label_counts.groupby("scope_status", as_index=False)["rows"].sum().sort_values("scope_status").reset_index(drop=True)
unfamiliar_or_changed_labels = sorted(raw_product_labels.difference(reference_2024_label_set))
familiar_out_of_scope_labels = sorted(raw_product_labels.intersection(reference_2024_label_set).difference(locked_class_set))

display(raw_product_label_counts.sort_values(["scope_status", "rows", "product_label"], ascending=[True, False, True]).reset_index(drop=True))
display(label_scope_summary)
print(f"Unfamiliar or changed product labels: {len(unfamiliar_or_changed_labels)}")
print(f"Familiar out-of-scope product labels: {len(familiar_out_of_scope_labels)}")
print("2025 structural validation: PASS")

,required_column,present
0,complaint_what_happened,True
1,product,True
2,date_received,True
3,complaint_id,True


,check,observed,expected
0,Total rows,50000,50000
1,Total columns,17,17
2,Minimum date_received,2025-01-01,2025-01-01
3,Maximum date_received,2025-12-31,2025-12-31
4,Rows outside 2025,0,0
5,Unparseable nonblank dates,0,0
6,Duplicate complaint_id values,0,0
7,Missing/blank narratives,0,0
8,Missing/blank product labels,0,0
9,Missing/blank date_received,0,0


,product_label,rows,scope_status
0,"Payday loan, title loan, personal loan, or adv...",403,familiar out-of-scope label
1,Prepaid card,238,familiar out-of-scope label
2,Debt or credit management,134,familiar out-of-scope label
3,Credit reporting or other personal consumer re...,35215,locked eight-category scope
4,Debt collection,5664,locked eight-category scope
5,Credit card,2369,locked eight-category scope
6,Checking or savings account,2216,locked eight-category scope
7,"Money transfer, virtual currency, or money ser...",1848,locked eight-category scope
8,Mortgage,720,locked eight-category scope
9,Vehicle loan or lease,646,locked eight-category scope


,scope_status,rows
0,familiar out-of-scope label,775
1,locked eight-category scope,49225


Unfamiliar or changed product labels: 0
Familiar out-of-scope product labels: 3
2025 structural validation: PASS


## Locked Version 1 Text Cleaning

The cleaning below reproduces the committed light-cleaning rules exactly. The cleaned narrative exists only in memory and is never displayed or saved.

In [11]:
URL_PATTERN_2025 = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
WHITESPACE_PATTERN_2025 = re.compile(r"\s+")


def clean_complaint_text_2025(value: object) -> str:
    text = "" if pd.isna(value) else str(value)
    text = URL_PATTERN_2025.sub(" ", text)
    text = WHITESPACE_PATTERN_2025.sub(" ", text)
    return text.strip()


rows_after_narrative_filter = ~narrative_blank_mask
sequential_missing_narratives = int(narrative_blank_mask.sum())
sequential_missing_products = int((rows_after_narrative_filter & product_blank_mask).sum())
required_field_eligible_mask = ~narrative_blank_mask & ~product_blank_mask

cleaning_source_2025 = raw_2025_df.loc[
    required_field_eligible_mask,
    ["complaint_what_happened", "product"],
].copy()
cleaning_source_2025["_source_order"] = np.flatnonzero(required_field_eligible_mask.to_numpy())
cleaning_source_2025["clean_complaint_text"] = cleaning_source_2025["complaint_what_happened"].map(clean_complaint_text_2025)
cleaning_source_2025["product"] = cleaning_source_2025["product"].astype("string").str.strip()

empty_after_cleaning_mask = cleaning_source_2025["clean_complaint_text"].str.len().eq(0) | cleaning_source_2025["product"].fillna("").str.len().eq(0)
rows_empty_after_cleaning = int(empty_after_cleaning_mask.sum())
cleaned_2025_df = cleaning_source_2025.loc[
    ~empty_after_cleaning_mask,
    ["_source_order", "clean_complaint_text", "product"],
].reset_index(drop=True)

record_check(
    "Locked 2025 URL-cleaning expression",
    URL_PATTERN_2025.pattern == r"https?://\S+|www\.\S+" and bool(URL_PATTERN_2025.flags & re.IGNORECASE),
    URL_PATTERN_2025.pattern,
    r"https?://\S+|www\.\S+ with IGNORECASE",
)
record_check(
    "Locked 2025 whitespace-cleaning expression",
    WHITESPACE_PATTERN_2025.pattern == r"\s+",
    WHITESPACE_PATTERN_2025.pattern,
    r"\s+",
)
cleaned_values_nonempty = bool(cleaned_2025_df["clean_complaint_text"].str.len().gt(0).all() and cleaned_2025_df["product"].str.len().gt(0).all())
record_check("2025 cleaned required values are nonempty", cleaned_values_nonempty, cleaned_values_nonempty, True)
stop_if_failed("Locked 2025 text cleaning")

cleaning_count_flow = pd.DataFrame(
    [
        {"stage": "Raw rows", "input_rows": len(raw_2025_df), "excluded_this_stage": 0, "remaining_rows": len(raw_2025_df)},
        {"stage": "Exclude missing/blank narrative", "input_rows": len(raw_2025_df), "excluded_this_stage": sequential_missing_narratives, "remaining_rows": int(rows_after_narrative_filter.sum())},
        {"stage": "Exclude missing/blank product", "input_rows": int(rows_after_narrative_filter.sum()), "excluded_this_stage": sequential_missing_products, "remaining_rows": int(required_field_eligible_mask.sum())},
        {"stage": "Exclude rows empty after locked cleaning", "input_rows": int(required_field_eligible_mask.sum()), "excluded_this_stage": rows_empty_after_cleaning, "remaining_rows": len(cleaned_2025_df)},
    ]
)
display(cleaning_count_flow)
print(f"Rows becoming empty after locked cleaning: {rows_empty_after_cleaning:,}")
print("Locked cleaning preserves case, punctuation, wording, and stop words; lowercasing remains inside the fitted vectorizer only.")
print("Locked 2025 text cleaning: PASS")

,stage,input_rows,excluded_this_stage,remaining_rows
0,Raw rows,50000,0,50000
1,Exclude missing/blank narrative,50000,0,50000
2,Exclude missing/blank product,50000,0,50000
3,Exclude rows empty after locked cleaning,50000,0,50000


Rows becoming empty after locked cleaning: 0
Locked cleaning preserves case, punctuation, wording, and stop words; lowercasing remains inside the fitted vectorizer only.
Locked 2025 text cleaning: PASS


## 2025 Duplicate and Cross-Year Overlap Audit

Normalized-text hashes are temporary in-memory grouping identifiers. Only aggregate group and affected-row counts are displayed.

In [12]:
cleaned_2025_df["normalized_text_hash"] = cleaned_2025_df["clean_complaint_text"].map(stable_text_hash)

overlap_development_mask_2025 = cleaned_2025_df["normalized_text_hash"].isin(development_groups)
overlap_final_test_mask_2025 = cleaned_2025_df["normalized_text_hash"].isin(final_test_groups)
overlap_either_mask_2025 = overlap_development_mask_2025 | overlap_final_test_mask_2025
overlap_2024_union_groups = development_groups.union(final_test_groups)

label_counts_by_hash_2025 = cleaned_2025_df.groupby("normalized_text_hash", sort=False)["product"].nunique()
conflicting_hashes_2025 = set(label_counts_by_hash_2025[label_counts_by_hash_2025 > 1].index)
conflicting_rows_mask_2025 = cleaned_2025_df["normalized_text_hash"].isin(conflicting_hashes_2025)

same_label_group_sizes_2025 = cleaned_2025_df.groupby(["normalized_text_hash", "product"], sort=False).size()
repeated_same_label_sizes_2025 = same_label_group_sizes_2025[same_label_group_sizes_2025 > 1]
repeated_same_label_group_count_2025 = int(len(repeated_same_label_sizes_2025))
repeated_same_label_rows_affected_2025 = int(repeated_same_label_sizes_2025.sum())

overlap_audit_table = pd.DataFrame(
    [
        {"audit": "Overlap with 2024 development", "unique_groups": int(cleaned_2025_df.loc[overlap_development_mask_2025, "normalized_text_hash"].nunique()), "affected_rows": int(overlap_development_mask_2025.sum())},
        {"audit": "Overlap with 2024 final internal test", "unique_groups": int(cleaned_2025_df.loc[overlap_final_test_mask_2025, "normalized_text_hash"].nunique()), "affected_rows": int(overlap_final_test_mask_2025.sum())},
        {"audit": "Overlap with either 2024 partition", "unique_groups": int(cleaned_2025_df.loc[overlap_either_mask_2025, "normalized_text_hash"].nunique()), "affected_rows": int(overlap_either_mask_2025.sum())},
    ]
)
duplicate_audit_table = pd.DataFrame(
    [
        {"audit": "Repeated same-label normalized-text groups", "groups": repeated_same_label_group_count_2025, "affected_rows": repeated_same_label_rows_affected_2025},
        {"audit": "Conflicting-label normalized-text groups", "groups": len(conflicting_hashes_2025), "affected_rows": int(conflicting_rows_mask_2025.sum())},
    ]
)
display(overlap_audit_table)
display(duplicate_audit_table)
print("No complaint narratives or normalized-text hashes were displayed or saved.")

,audit,unique_groups,affected_rows
0,Overlap with 2024 development,161,3772
1,Overlap with 2024 final internal test,42,1807
2,Overlap with either 2024 partition,203,5579


,audit,groups,affected_rows
0,Repeated same-label normalized-text groups,5033,23843
1,Conflicting-label normalized-text groups,37,1660


No complaint narratives or normalized-text hashes were displayed or saved.


## Committed 2025 Cohorts

The secondary cohort retains every otherwise eligible locked-scope row. The primary cohort applies the committed exclusions in their fixed order: cross-year overlap, 2025 conflicting-label groups identified before category filtering, then repeated same-label deduplication keeping the first original row.

In [13]:
secondary_2025_df = cleaned_2025_df.loc[cleaned_2025_df["product"].isin(EXPECTED_CLASSES)].copy()
out_of_scope_rows_2025 = len(cleaned_2025_df) - len(secondary_2025_df)

primary_after_overlap_2025 = secondary_2025_df.loc[
    ~secondary_2025_df["normalized_text_hash"].isin(overlap_2024_union_groups)
].copy()
primary_overlap_rows_excluded_2025 = len(secondary_2025_df) - len(primary_after_overlap_2025)

primary_after_conflicts_2025 = primary_after_overlap_2025.loc[
    ~primary_after_overlap_2025["normalized_text_hash"].isin(conflicting_hashes_2025)
].copy()
primary_conflicting_rows_excluded_2025 = len(primary_after_overlap_2025) - len(primary_after_conflicts_2025)

primary_2025_df = primary_after_conflicts_2025.drop_duplicates(
    subset=["normalized_text_hash", "product"],
    keep="first",
).reset_index(drop=True)
primary_repeated_rows_removed_2025 = len(primary_after_conflicts_2025) - len(primary_2025_df)

primary_hashes_unique = bool(primary_2025_df["normalized_text_hash"].is_unique)
primary_has_no_conflicts = not bool(primary_2025_df["normalized_text_hash"].isin(conflicting_hashes_2025).any())
primary_has_no_2024_overlap = not bool(primary_2025_df["normalized_text_hash"].isin(overlap_2024_union_groups).any())
primary_has_only_locked_categories = set(primary_2025_df["product"].unique()).issubset(locked_class_set)
secondary_retains_all_locked_eligible_rows = len(secondary_2025_df) == int(cleaned_2025_df["product"].isin(EXPECTED_CLASSES).sum())

record_check("Primary cohort has one row per normalized-text hash", primary_hashes_unique, primary_hashes_unique, True)
record_check("Primary cohort has no conflicting-label groups", primary_has_no_conflicts, primary_has_no_conflicts, True)
record_check("Primary cohort has no 2024 partition overlap", primary_has_no_2024_overlap, primary_has_no_2024_overlap, True)
record_check("Primary cohort uses only locked categories", primary_has_only_locked_categories, primary_has_only_locked_categories, True)
record_check("Secondary cohort retains all eligible locked-scope rows", secondary_retains_all_locked_eligible_rows, secondary_retains_all_locked_eligible_rows, True)
stop_if_failed("Committed 2025 cohort construction")

secondary_count_flow = pd.DataFrame(
    [
        {"stage": "Raw 2025 rows", "input_rows": len(raw_2025_df), "excluded_this_stage": 0, "remaining_rows": len(raw_2025_df)},
        {"stage": "Exclude missing/blank narratives", "input_rows": len(raw_2025_df), "excluded_this_stage": sequential_missing_narratives, "remaining_rows": int(rows_after_narrative_filter.sum())},
        {"stage": "Exclude missing/blank products", "input_rows": int(rows_after_narrative_filter.sum()), "excluded_this_stage": sequential_missing_products, "remaining_rows": int(required_field_eligible_mask.sum())},
        {"stage": "Exclude rows empty after cleaning", "input_rows": int(required_field_eligible_mask.sum()), "excluded_this_stage": rows_empty_after_cleaning, "remaining_rows": len(cleaned_2025_df)},
        {"stage": "Restrict to locked eight-category scope", "input_rows": len(cleaned_2025_df), "excluded_this_stage": out_of_scope_rows_2025, "remaining_rows": len(secondary_2025_df)},
        {"stage": "Secondary operational cohort", "input_rows": len(secondary_2025_df), "excluded_this_stage": 0, "remaining_rows": len(secondary_2025_df)},
    ]
)
primary_count_flow = pd.DataFrame(
    [
        {"stage": "Start with secondary locked-scope rows", "input_rows": len(secondary_2025_df), "excluded_this_stage": 0, "remaining_rows": len(secondary_2025_df)},
        {"stage": "Exclude overlap with either 2024 partition", "input_rows": len(secondary_2025_df), "excluded_this_stage": primary_overlap_rows_excluded_2025, "remaining_rows": len(primary_after_overlap_2025)},
        {"stage": "Exclude 2025 conflicting-label groups", "input_rows": len(primary_after_overlap_2025), "excluded_this_stage": primary_conflicting_rows_excluded_2025, "remaining_rows": len(primary_after_conflicts_2025)},
        {"stage": "Keep first remaining same-label text", "input_rows": len(primary_after_conflicts_2025), "excluded_this_stage": primary_repeated_rows_removed_2025, "remaining_rows": len(primary_2025_df)},
        {"stage": "Primary leakage-resistant cohort", "input_rows": len(primary_2025_df), "excluded_this_stage": 0, "remaining_rows": len(primary_2025_df)},
    ]
)
display(secondary_count_flow)
display(primary_count_flow)
print(f"Secondary operational cohort rows: {len(secondary_2025_df):,}")
print(f"Primary leakage-resistant cohort rows: {len(primary_2025_df):,}")
print("Committed 2025 cohort construction: PASS")

,stage,input_rows,excluded_this_stage,remaining_rows
0,Raw 2025 rows,50000,0,50000
1,Exclude missing/blank narratives,50000,0,50000
2,Exclude missing/blank products,50000,0,50000
3,Exclude rows empty after cleaning,50000,0,50000
4,Restrict to locked eight-category scope,50000,775,49225
5,Secondary operational cohort,49225,0,49225


,stage,input_rows,excluded_this_stage,remaining_rows
0,Start with secondary locked-scope rows,49225,0,49225
1,Exclude overlap with either 2024 partition,49225,5579,43646
2,Exclude 2025 conflicting-label groups,43646,1301,42345
3,Keep first remaining same-label text,42345,12189,30156
4,Primary leakage-resistant cohort,30156,0,30156


Secondary operational cohort rows: 49,225
Primary leakage-resistant cohort rows: 30,156
Committed 2025 cohort construction: PASS


## 2025 Structural, Cleaning, Audit, and Cohort Summary

This final safety audit confirms that the newly authorized cells contain no fitting, prediction, decision-score, metric, figure, or row-level export operations.

In [14]:
with NOTEBOOK_PATH.open("r", encoding="utf-8") as notebook_handle:
    final_notebook_document = nbformat.read(notebook_handle, as_version=4)

authorized_2025_cell_ids = {
    "load-2025-structural",
    "clean-2025",
    "audit-2025-overlap",
    "cohorts-2025",
}
authorized_2025_sources = [
    cell.source
    for cell in final_notebook_document.cells
    if cell.cell_type == "code" and cell.id in authorized_2025_cell_ids
]
forbidden_2025_model_calls = []
forbidden_2025_save_calls = []
sensitive_2025_display_calls = []
observed_2025_read_csv_arguments = []
forbidden_model_call_names = {"fit", "fit_transform", "predict", "predict_proba", "decision_function", "score"}
forbidden_save_call_names = {"to_csv", "to_parquet", "to_pickle", "to_json", "save", "savez", "dump"}
sensitive_variable_names = {
    "raw_2025_df",
    "cleaning_source_2025",
    "cleaned_2025_df",
    "secondary_2025_df",
    "primary_2025_df",
    "development_groups",
    "final_test_groups",
    "conflicting_hashes_2025",
}

for source in authorized_2025_sources:
    syntax_tree = ast.parse(source)
    for node in ast.walk(syntax_tree):
        if not isinstance(node, ast.Call):
            continue
        if isinstance(node.func, ast.Attribute):
            call_name = node.func.attr
        elif isinstance(node.func, ast.Name):
            call_name = node.func.id
        else:
            call_name = ""
        if call_name in forbidden_model_call_names:
            forbidden_2025_model_calls.append(call_name)
        if call_name in forbidden_save_call_names:
            forbidden_2025_save_calls.append(call_name)
        if call_name == "read_csv" and node.args:
            observed_2025_read_csv_arguments.append(ast.unparse(node.args[0]))
        if call_name == "display" and node.args:
            displayed_expression = ast.unparse(node.args[0])
            if any(name in displayed_expression for name in sensitive_variable_names):
                sensitive_2025_display_calls.append(displayed_expression)

record_check("No 2025 fitting, prediction, or decision-score call", not forbidden_2025_model_calls, forbidden_2025_model_calls or "none", "none")
record_check("Exactly one locked 2025 CSV load", observed_2025_read_csv_arguments == ["HOLDOUT_2025_PATH"], observed_2025_read_csv_arguments, ["HOLDOUT_2025_PATH"])
record_check("No 2025 row-level export or save call", not forbidden_2025_save_calls, forbidden_2025_save_calls or "none", "none")
record_check("No sensitive 2025 dataframe display call", not sensitive_2025_display_calls, sensitive_2025_display_calls or "none", "none")
stop_if_failed("2025 data preparation and cohort safety audit")

phase2_data_check_table = pd.DataFrame(check_records[phase2_data_check_start:])
display(phase2_data_check_table)
complete_check_table = pd.DataFrame(check_records)
all_current_checks_passed = bool(complete_check_table["status"].eq("PASS").all())
if not all_current_checks_passed:
    raise RuntimeError("The 2025 structural, cleaning, audit, or cohort gate FAILED. Do not evaluate the model.")

print("Previous Phase 2 entry checks passed: 62/62")
print(f"New structural/cleaning/cohort checks passed: {len(phase2_data_check_table)}/{len(phase2_data_check_table)}")
print(f"All current notebook checks passed: {len(complete_check_table)}/{len(complete_check_table)}")
print("2025 STRUCTURAL, CLEANING, DUPLICATE, OVERLAP, AND COHORT GATE: PASS")
print("No 2025 model predictions, decision scores, classification metrics, routing metrics, drift metrics, or figures were produced.")

,check,status,observed,expected
0,2025 required raw columns are present,PASS,all present,all present
1,2025 raw row count,PASS,50000,50000
2,2025 raw column count,PASS,17,17
3,2025 minimum date_received,PASS,2025-01-01,2025-01-01
4,2025 maximum date_received,PASS,2025-12-31,2025-12-31
5,2025 rows outside calendar year,PASS,0,0
6,2025 unparseable nonblank dates,PASS,0,0
7,2025 duplicate complaint_id values,PASS,0,0
8,2025 missing or blank narratives,PASS,0,0
9,2025 missing or blank product labels,PASS,0,0


Previous Phase 2 entry checks passed: 62/62
New structural/cleaning/cohort checks passed: 24/24
All current notebook checks passed: 86/86
2025 STRUCTURAL, CLEANING, DUPLICATE, OVERLAP, AND COHORT GATE: PASS
No 2025 model predictions, decision scores, classification metrics, routing metrics, drift metrics, or figures were produced.
